# Demo 06 : Debugging with stream modes

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.5", "langchain-core==1.6.10", "langchain-openai==1.7.1", "langgraph==1.2.15", "langgraph-checkpoint-sqlite==3.1.2", "deepagents==0.7.24", "langsmith==0.14.8", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    THINKING_OFF = {"reasoning": {"effort": "none"}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# New OpenRouter accounts may send 20 requests a minute per model.
# One shared limiter keeps parallel agents under it ; raise COURSE_RPS when the cap lifts
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "0.3")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    kw.setdefault("temperature", 0)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · stream-modes

We port ARI510's streaming modes and inspect state alongside a live model reply. A checkpoint saver enables checkpoint events.

In [2]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.config import get_stream_writer

class DebugState(TypedDict):
    report: str
    reply: str

def acknowledge(state: DebugState):
    get_stream_writer()({"status": "drafting"})
    return {"reply": llm.invoke("Write one brief acknowledgement of this bus service report: " + state["report"]).content}

builder = StateGraph(DebugState)
builder.add_node("acknowledge", acknowledge)
builder.add_edge(START, "acknowledge")
builder.add_edge("acknowledge", END)
graph = builder.compile(checkpointer=InMemorySaver())
inputs = {"report": "Bus 4012 needs brake inspection at Noord.", "reply": ""}
config = {"configurable": {"thread_id": "debug-live"}}
modes = ["values", "updates", "debug", "messages", "custom", "checkpoints", "tasks"]
parts = list(graph.stream(inputs, config, stream_mode=modes, version="v2"))
for part in parts:
    if part["type"] == "messages":
        continue
    if part["type"] in {"debug", "tasks", "checkpoints"}:
        print(part["type"], ":", str(part["data"])[:500])
    else:
        print(part["type"], ":", part["data"])
tokens = [part["data"][0].content for part in parts if part["type"] == "messages"]
print("message chunks :", len(tokens))
print("reply :", graph.get_state(config).values["reply"])
print("v2 keys :", sorted(parts[0]))
print("namespace :", parts[0]["ns"])
print("emitted modes :", sorted({part["type"] for part in parts}))

checkpoints : {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'debug-live', 'checkpoint_id': '1f1c74aa-b2b1-6145-bfff-78cb06be52d1'}}, 'parent_config': None, 'values': {}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': 'e1d9fee3-68e9-26ad-d643-33e8afd53b27', 'name': '__start__', 'interrupts': (), 'state': None}]}
debug : {'step': -1, 'timestamp': '2026-10-13T21:11:33.842976+00:00', 'type': 'checkpoint', 'payload': {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'debug-live', 'checkpoint_id': '1f1c74aa-b2b1-6145-bfff-78cb06be52d1'}}, 'parent_config': None, 'values': {}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': 'e1d9fee3-68e9-26ad-d643-33e8afd53b27', 'name': '__start__', 'interrupts': (), 'state': None}]}}
values : {'report': 'Bus 4012 needs brake inspection at Noord.', 'reply': ''}
checkpoints : {'config': {'configurable': {'checkpoint_ns': '', 't

We inspect experimental v3 projections on a deterministic graph to avoid another model call. Iterating a projection drives the run; output becomes available when the run finishes.

In [3]:
class Counter(TypedDict):
    value: int

small = StateGraph(Counter)
small.add_node("increment", lambda state: {"value": state["value"] + 1})
small.add_edge(START, "increment")
small.add_edge("increment", END)
small_graph = small.compile()
run = small_graph.stream_events({"value": 0}, version="v3")
for state in run.values:
    print("v3 value :", state)
print("v3 output :", run.output)
print("v3 interrupted :", run.interrupted)
parent = StateGraph(Counter)
parent.add_node("counter", small_graph)
parent.add_edge(START, "counter")
parent.add_edge("counter", END)
lifecycle = parent.compile().stream_events({"value": 0}, version="v3")
for event in lifecycle.lifecycle:
    print("v3 lifecycle :", event)

v3 value : {'value': 0}
v3 value : {'value': 1}
v3 output : {'value': 1}
v3 interrupted : False
v3 lifecycle : {'event': 'started', 'namespace': ['counter:0461b8f3-5710-bba6-a08b-703410a23696'], 'graph_name': 'counter', 'trigger_call_id': '0461b8f3-5710-bba6-a08b-703410a23696'}
v3 lifecycle : {'event': 'completed', 'namespace': ['counter:0461b8f3-5710-bba6-a08b-703410a23696']}


## Things to try

Optional : we can change a ticket, a thread id or a returned update and compare the saved state.